# AI Programming — Lecture 4
## 로지스틱 회귀 (Logistic Regression)

이 노트북에서는 **분류(classification)**의 기본 개념부터
**binary logistic regression**, **softmax**, **cross entropy**,
그리고 **classification metrics**까지 단계적으로 실습합니다.

### 학습 목표
실습을 마치면 다음 내용을 설명하고 구현할 수 있어야 합니다.

- regression과 classification의 차이를 설명할 수 있습니다.
- 분류 모델이 class probability를 출력하는 이유를 이해합니다.
- sigmoid 함수와 logistic regression의 관계를 설명할 수 있습니다.
- binary cross entropy(BCE)를 직접 계산할 수 있습니다.
- gradient descent로 logistic regression을 직접 학습할 수 있습니다.
- TensorFlow/Keras로 binary logistic regression을 구현할 수 있습니다.
- softmax와 categorical cross entropy(CCE)를 계산할 수 있습니다.
- entropy와 cross entropy의 의미를 비교할 수 있습니다.
- confusion matrix에서 TP, TN, FP, FN을 구할 수 있습니다.
- accuracy, precision, recall, F1 score를 계산할 수 있습니다.
- imbalanced data에서 accuracy만 사용하는 것이 왜 위험한지 설명할 수 있습니다.

### 실습 방법
1. 셀을 **위에서부터 순서대로 실행**하세요.
2. 각 절의 **확인할 내용**을 읽고 출력과 그래프를 해석하세요.
3. `TODO`가 있는 부분은 값을 직접 변경해 다시 실행하세요.
4. 분류에서는 **확률(probability)**과 **최종 class prediction**을 구분해서 확인하세요.

> 그래프의 축과 제목은 실행 환경의 한글 폰트 문제를 피하기 위해 일부 영어로 표시합니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# Part I. Classification

## 1. Regression vs. Classification

Machine learning model은 입력 $\mathbf{x}$를 받아 어떤 출력 $\hat{y}$를 만듭니다.

### Regression
연속적인 값을 예측합니다.

예:
- 주택 가격
- 온도
- 시험 점수

### Classification
여러 class 중 하나를 선택합니다.

예:
- cat / dog / wolf / fox
- spam / not spam
- pass / fail

분류에서는 class를 단순한 숫자의 크기나 거리로 해석하지 않고,
각 class에 대한 **확률 또는 score**를 출력하도록 구성합니다.

## 2. 분류 label과 확률 출력

4개의 class가 있다고 가정하면 하나의 정답은 one-hot vector로 표현할 수 있습니다.

예를 들어 class 1이 정답이라면

$$
\mathbf{y}
=
[0,\ 1,\ 0,\ 0]
$$

모델은 다음과 같은 class probability를 출력할 수 있습니다.

$$
\hat{\mathbf{y}}
=
[0.02,\ 0.85,\ 0.10,\ 0.03]
$$

가장 확률이 큰 class를 최종 prediction으로 선택할 수 있습니다.

In [ ]:
classes = np.array(["cat", "dog", "wolf", "fox"])

y_true = np.array([0, 1, 0, 0])
y_prob = np.array([0.02, 0.85, 0.10, 0.03])

pred_index = np.argmax(y_prob)
pred_class = classes[pred_index]

print("정답 one-hot:", y_true)
print("예측 확률:", y_prob)
print("확률의 합:", y_prob.sum())
print("예측 class:", pred_class)

### 확인할 내용

- class probability의 합은 1입니다.
- `argmax`는 가장 큰 확률을 가진 class를 선택합니다.
- label `0, 1, 2, 3` 자체의 숫자 크기는 class 간 거리를 의미하지 않습니다.

## 3. Softmax: scores를 probabilities로 바꾸기

모델의 마지막 layer는 먼저 class별 score인 **logits**를 만들 수 있습니다.

Softmax는 logits $\mathbf{z}$를 확률 벡터 $\hat{\mathbf{y}}$로 변환합니다.

$$
\hat{y}_i
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$

각 원소는 0과 1 사이이고 전체 합은 1입니다.

In [ ]:
def softmax(z):
    z = np.asarray(z, dtype=float)

    # numerical stability
    z = z - np.max(z)

    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)


logits = np.array([1.0, 3.0, 2.0, 0.5])
probs = softmax(logits)

print("logits:", logits)
print("probabilities:", probs)
print("sum:", probs.sum())
print("predicted index:", np.argmax(probs))

### 직접 해보기 1 — logits 바꾸기

`logits`의 한 값을 크게 바꾸어 보세요.

예:

```python
logits = np.array([1.0, 8.0, 2.0, 0.5])
```

### 질문
1. 가장 큰 logit의 class probability는 어떻게 변합니까?
2. 다른 class probability는 어떻게 변합니까?
3. 전체 합은 여전히 1입니까?

### Language Model과 같은 구조

Language Model에서도 다음 token을 하나의 class로 생각할 수 있습니다.

예를 들어 vocabulary가 다음 네 token만 가진다고 가정해 봅니다.

```text
["마음은", "학교는", "때에는", "때마다"]
```

모델은 각 token의 logit을 만들고 softmax를 이용해 probability distribution을 생성합니다.

In [ ]:
tokens = np.array(["마음은", "학교는", "때에는", "때마다"])

token_logits = np.array([0.2, 0.5, 4.0, 1.0])
token_probs = softmax(token_logits)

for token, p in zip(tokens, token_probs):
    print(f"{token:>5}: {p:.4f}")

print("\nGreedy decoding:", tokens[np.argmax(token_probs)])

# Part II. Logistic Regression

## 4. Binary Classification Example

강의와 같은 단순한 예제를 사용합니다.

| Study Time | 1 | 3 | 5 | 7 | 9 | 11 | 13 |
|---|---:|---:|---:|---:|---:|---:|---:|
| Outcome | 0 | 0 | 0 | 1 | 1 | 1 | 1 |

여기서

- Fail = 0
- Pass = 1

입니다.

Logistic regression은 이름에 `regression`이 들어가지만
**binary classification**에 사용됩니다.

In [ ]:
hours = np.array([1, 3, 5, 7, 9, 11, 13], dtype=float)
outcome = np.array([0, 0, 0, 1, 1, 1, 1], dtype=float)

plt.scatter(hours, outcome)
plt.xlabel("Study Time (hours)")
plt.ylabel("Outcome")
plt.yticks([0, 1], ["Fail", "Pass"])
plt.title("Binary Classification Data")
plt.grid(alpha=0.3)
plt.show()

## 5. 왜 Sigmoid가 필요한가?

Linear regression은

$$
z = ax + b
$$

처럼 제한되지 않은 실수값을 출력합니다.

하지만 binary classification에서 원하는 출력은
**0과 1 사이의 probability**입니다.

Sigmoid function은

$$
\sigma(z)
=
\frac{1}{1+e^{-z}}
$$

이며 어떤 실수 $z$도 $(0,1)$ 범위로 변환합니다.

In [ ]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-z))


z = np.linspace(-10, 10, 300)

plt.plot(z, sigmoid(z))
plt.axhline(0.5, linestyle="--")
plt.axvline(0.0, linestyle="--")
plt.xlabel("z")
plt.ylabel("sigmoid(z)")
plt.title("Sigmoid Function")
plt.grid(alpha=0.3)
plt.show()

### Logistic Regression Model

Logistic regression은

$$
z = ax+b
$$

를 먼저 계산한 뒤 sigmoid를 통과시킵니다.

$$
\hat{y}
=
\sigma(ax+b)
$$

여기서

- $a$: weight
- $b$: bias
- $\hat{y}$: class 1일 probability

입니다.

In [ ]:
def logistic_probability(x, a, b):
    return sigmoid(a * x + b)


x_line = np.linspace(0, 14, 300)

# 아직 학습하지 않은 예시 파라미터
a_demo = 1.0
b_demo = -6.0

p_demo = logistic_probability(x_line, a_demo, b_demo)

plt.scatter(hours, outcome, label="Data")
plt.plot(x_line, p_demo, label="Sigmoid curve")
plt.axhline(0.5, linestyle="--", label="Threshold = 0.5")
plt.xlabel("Study Time (hours)")
plt.ylabel("Pass Probability")
plt.title("Logistic Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- $a$는 sigmoid curve가 얼마나 빠르게 증가하거나 감소하는지에 영향을 줍니다.
- $b$는 sigmoid curve의 위치를 좌우로 이동시킵니다.
- probability가 `0.5`가 되는 위치는 decision boundary로 볼 수 있습니다.

### 직접 해보기 2 — Sigmoid shape 바꾸기

아래 값들을 바꾸어 그래프를 다시 그려 보세요.

- `a = 0.5`, `1.0`, `3.0`
- `b = -3.0`, `-6.0`, `-9.0`

`a`와 `b`가 각각 sigmoid curve에 어떤 영향을 주는지 확인하세요.

In [ ]:
# TODO: 값을 바꾸어 보세요.
a_try = 1.0
b_try = -6.0

p_try = logistic_probability(x_line, a_try, b_try)

plt.scatter(hours, outcome)
plt.plot(x_line, p_try)
plt.axhline(0.5, linestyle="--")
plt.xlabel("Study Time (hours)")
plt.ylabel("Pass Probability")
plt.title(f"a={a_try}, b={b_try}")
plt.grid(alpha=0.3)
plt.show()

## 6. Binary Cross Entropy (BCE)

Binary classification에서는 대표적으로 BCE를 사용합니다.

정답이 $y=1$일 때:

$$
\mathcal{L}
=
-\log \hat{y}
$$

정답이 $y=0$일 때:

$$
\mathcal{L}
=
-\log(1-\hat{y})
$$

두 경우를 하나의 식으로 쓰면

$$
\mathcal{L}
=
-
\left[
y\log\hat{y}
+
(1-y)\log(1-\hat{y})
\right]
$$

입니다.

In [ ]:
def binary_cross_entropy(y_true, y_prob, eps=1e-7):
    y_prob = np.clip(y_prob, eps, 1.0 - eps)

    return -np.mean(
        y_true * np.log(y_prob)
        + (1.0 - y_true) * np.log(1.0 - y_prob)
    )


print("정답 1, 예측 0.9 :", binary_cross_entropy(
    np.array([1.0]),
    np.array([0.9])
))

print("정답 1, 예측 0.1 :", binary_cross_entropy(
    np.array([1.0]),
    np.array([0.1])
))

print("정답 0, 예측 0.1 :", binary_cross_entropy(
    np.array([0.0]),
    np.array([0.1])
))

print("정답 0, 예측 0.9 :", binary_cross_entropy(
    np.array([0.0]),
    np.array([0.9])
))

> ### ✅ 체크포인트
> 정답과 같은 방향으로 높은 확률을 주면 loss가 작습니다.  
> 반대로 **틀린 class에 높은 확률을 줄수록 loss가 크게 증가**합니다.

## 7. Logistic Regression 직접 학습하기

앞에서 정의한

$$
\hat{y}_i
=
\sigma(ax_i+b)
$$

와 BCE를 이용해 $a$, $b$를 gradient descent로 학습합니다.

BCE와 sigmoid를 함께 사용하면 gradient는 다음과 같이 단순해집니다.

$$
\frac{\partial \mathcal{L}}{\partial a}
=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat{y}_i-y_i)x_i
$$

$$
\frac{\partial \mathcal{L}}{\partial b}
=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat{y}_i-y_i)
$$

In [ ]:
a = 0.0
b = 0.0

lr = 0.05
epochs = 3000

loss_history = []

for epoch in range(epochs):
    # 1) probability prediction
    y_prob = sigmoid(a * hours + b)

    # 2) BCE loss
    loss = binary_cross_entropy(outcome, y_prob)
    loss_history.append(loss)

    # 3) gradients
    error = y_prob - outcome

    grad_a = np.mean(error * hours)
    grad_b = np.mean(error)

    # 4) parameter update
    a -= lr * grad_a
    b -= lr * grad_b

    if epoch % 500 == 0:
        print(
            f"epoch={epoch:4d}, "
            f"a={a:.4f}, "
            f"b={b:.4f}, "
            f"BCE={loss:.4f}"
        )

print("\n최종 모델")
print(f"a = {a:.4f}")
print(f"b = {b:.4f}")

In [ ]:
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Training Loss")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
x_line = np.linspace(0, 14, 300)
p_line = sigmoid(a * x_line + b)

plt.scatter(hours, outcome, label="Data")
plt.plot(x_line, p_line, label="Learned probability")
plt.axhline(0.5, linestyle="--", label="Threshold = 0.5")
plt.xlabel("Study Time (hours)")
plt.ylabel("Pass Probability")
plt.title("Learned Logistic Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- epoch가 증가하면서 BCE가 감소하는지 확인하세요.
- 학습된 sigmoid curve가 fail과 pass 데이터를 잘 나누는지 확인하세요.
- probability가 0.5인 지점이 대략 어느 study time에 위치하는지 확인하세요.

### 직접 해보기 3 — 새로운 입력 예측

아래 `new_hours` 값을 바꾸어 합격 probability를 확인하세요.

In [ ]:
# TODO: 값을 바꾸어 보세요.
new_hours = 8.0

pass_prob = sigmoid(a * new_hours + b)
predicted_class = int(pass_prob >= 0.5)

print(f"Study Time: {new_hours} hours")
print(f"Pass probability: {pass_prob:.4f}")
print(f"Predicted class: {predicted_class}")

# Part III. TensorFlow/Keras Implementation

## 8. Keras로 Binary Logistic Regression 구현하기

강의 슬라이드의 구조와 동일하게
`Dense(1, activation="sigmoid")`를 사용합니다.

이 layer는 내부적으로

$$
z = ax+b
$$

를 계산하고 sigmoid를 적용합니다.

즉, 입력에 곱해지는 weight가 $a$,
bias가 $b$에 해당합니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense

tf.random.set_seed(0)

x_tf = hours.astype(np.float32).reshape(-1, 1)
y_tf = outcome.astype(np.float32)

model = Sequential([
    Input(shape=(1,)),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.05),
    loss="binary_crossentropy"
)

history = model.fit(
    x_tf,
    y_tf,
    epochs=1500,
    verbose=0
)

print("final loss:", history.history["loss"][-1])

In [ ]:
# 학습된 probability
y_prob_tf = model(
    x_tf,
    training=False
).numpy().squeeze()

# 학습된 weight와 bias
weight, bias = model.layers[0].get_weights()

print("학습된 weight:", weight.squeeze())
print("학습된 bias:", bias.squeeze())
print("probabilities:", y_prob_tf)

In [ ]:
x_plot = np.linspace(0, 14, 200, dtype=np.float32).reshape(-1, 1)

p_plot = model(
    x_plot,
    training=False
).numpy().squeeze()

plt.scatter(hours, outcome)
plt.plot(x_plot.squeeze(), p_plot)
plt.axhline(0.5, linestyle="--")
plt.xlabel("Study Time (hours)")
plt.ylabel("Pass Probability")
plt.title("Keras Logistic Regression")
plt.grid(alpha=0.3)
plt.show()

# TODO: 값을 바꾸어 보세요.
new_hours = 8.0

sample = np.array([[new_hours]], dtype=np.float32)

prediction = model(
    sample,
    training=False
).numpy().squeeze()

print(
    f"{new_hours}시간 공부하면 "
    f"합격 확률은 {float(prediction):.4f}입니다."
)

### 확인할 내용

- Keras가 학습한 `weight`, `bias`가 앞에서 직접 학습한 $a$, $b$와 같은 역할을 합니다.
- 확률은 실행할 때마다 초기값과 optimizer에 따라 약간 달라질 수 있습니다.

# Part IV. Softmax Regression과 Cross Entropy

## 9. Multiclass Classification

여러 class를 분류할 때는 각 class별 logit을 만들고 softmax를 적용합니다.

$$
\hat{y}_c
=
\frac{e^{z_c}}
{\sum_j e^{z_j}}
$$

정답이 one-hot vector라면 categorical cross entropy(CCE)는

$$
\mathcal{L}
=
-
\sum_{c=1}^{C}
y_c \log \hat{y}_c
$$

입니다.

In [ ]:
y_true_onehot = np.array([0, 1, 0, 0], dtype=float)
y_pred_prob = np.array([0.02, 0.85, 0.10, 0.03], dtype=float)

cce = -np.sum(
    y_true_onehot
    * np.log(y_pred_prob)
)

print("CCE:", cce)
print("정답 class probability:", y_pred_prob[1])
print("-log(correct probability):", -np.log(y_pred_prob[1]))

### 확인할 내용

one-hot target에서는 CCE가 사실상

**정답 class에 할당한 probability의 negative log**

와 같습니다.

## 10. Keras Softmax Output 예제

강의 슬라이드의 확장 예제처럼
hidden Dense layer 뒤에 2개의 softmax output을 사용해 보겠습니다.

여기서는

- class 0 = Fail
- class 1 = Pass

입니다.

In [ ]:
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(0)

y_categorical = to_categorical(
    outcome.astype(int),
    num_classes=2
)

model_softmax = Sequential([
    Input(shape=(1,)),
    Dense(10, activation="relu"),
    Dense(2, activation="softmax")
])

model_softmax.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_softmax = model_softmax.fit(
    x_tf,
    y_categorical,
    epochs=2000,
    verbose=0
)

print("final loss:", history_softmax.history["loss"][-1])
print("final accuracy:", history_softmax.history["accuracy"][-1])
print("parameters:", model_softmax.count_params())

In [ ]:
prob_softmax = model_softmax(
    x_tf,
    training=False
).numpy()

print("class probabilities:")
print(prob_softmax)

print("\nrow sums:")
print(prob_softmax.sum(axis=1))

In [ ]:
x_plot = np.linspace(0, 14, 200, dtype=np.float32).reshape(-1, 1)

prob_plot = model_softmax(
    x_plot,
    training=False
).numpy()

plt.scatter(hours, outcome, label="Data")
plt.plot(x_plot.squeeze(), prob_plot[:, 0], label="Fail probability")
plt.plot(x_plot.squeeze(), prob_plot[:, 1], label="Pass probability")
plt.xlabel("Study Time (hours)")
plt.ylabel("Probability")
plt.title("Softmax Output")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

각 입력에 대해

```python
P(Fail) + P(Pass) = 1
```

이 되는지 확인하세요.

두 probability curve가 만나는 지점은
두 class가 같은 probability를 가지는 지점입니다.

# Part V. Entropy와 Cross Entropy

## 11. Entropy

Probability distribution $p$의 entropy는

$$
H(p)
=
-
\sum_i p_i \log p_i
$$

입니다.

불확실성이 클수록 entropy가 커집니다.

In [ ]:
def entropy(p, eps=1e-12):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log2(p))


fair_coin = np.array([0.5, 0.5])
biased_coin = np.array([0.99, 0.01])

fair_dice = np.ones(6) / 6

print("Fair coin entropy:", entropy(fair_coin), "bits")
print("Biased coin entropy:", entropy(biased_coin), "bits")
print("Fair dice entropy:", entropy(fair_dice), "bits")

### 확인할 내용

- 결과가 거의 확실한 distribution은 entropy가 낮습니다.
- 여러 결과가 비슷한 probability를 가지면 entropy가 높습니다.

## 12. Cross Entropy

Cross entropy는 target distribution $p$와
model distribution $q$ 사이의 mismatch를 측정합니다.

$$
H(p,q)
=
-
\sum_i p_i \log q_i
$$

Machine learning에서는

- $p$: ground truth
- $q$: model prediction

으로 생각할 수 있습니다.

In [ ]:
def cross_entropy(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    q = np.clip(q, eps, 1.0)

    return -np.sum(p * np.log(q))


target = np.array([0, 1, 0], dtype=float)

good_prediction = np.array([0.05, 0.90, 0.05])
bad_prediction = np.array([0.45, 0.10, 0.45])

print(
    "Good prediction CE:",
    cross_entropy(target, good_prediction)
)

print(
    "Bad prediction CE :",
    cross_entropy(target, bad_prediction)
)

# Part VI. Performance Metrics for Classification

## 13. Confusion Matrix

Binary classification에서는 prediction과 actual label을 비교해
다음 네 경우를 계산합니다.

| | Actual Positive | Actual Negative |
|---|---:|---:|
| Predicted Positive | TP | FP |
| Predicted Negative | FN | TN |

- **TP**: positive를 positive로 맞춤
- **TN**: negative를 negative로 맞춤
- **FP**: negative인데 positive라고 예측
- **FN**: positive인데 negative라고 예측

In [ ]:
def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    return tp, tn, fp, fn


example_true = np.array([1, 1, 1, 0, 0, 0, 0, 1])
example_pred = np.array([1, 0, 1, 0, 1, 0, 0, 1])

tp, tn, fp, fn = confusion_counts(
    example_true,
    example_pred
)

print("TP =", tp)
print("TN =", tn)
print("FP =", fp)
print("FN =", fn)

## 14. Accuracy, Precision, Recall, F1 Score

### Accuracy

$$
\mathrm{Accuracy}
=
\frac{TP+TN}
{TP+TN+FP+FN}
$$

### Precision

$$
\mathrm{Precision}
=
\frac{TP}
{TP+FP}
$$

### Recall

$$
\mathrm{Recall}
=
\frac{TP}
{TP+FN}
$$

### F1 Score

$$
F_1
=
2
\frac{
\mathrm{Precision}\cdot\mathrm{Recall}
}{
\mathrm{Precision}+\mathrm{Recall}
}
$$

In [ ]:
def classification_metrics(tp, tn, fp, fn):
    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (
            np.isfinite(precision)
            and np.isfinite(recall)
            and (precision + recall) > 0
        )
        else np.nan
    )

    return accuracy, precision, recall, f1


accuracy, precision, recall, f1 = classification_metrics(
    tp, tn, fp, fn
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

## 15. Classification Threshold

Binary classifier가 probability를 출력하면
threshold를 정해 최종 class를 결정합니다.

보통 다음처럼 사용합니다.

```python
prediction = (probability >= 0.5)
```

하지만 threshold를 바꾸면 precision과 recall도 달라집니다.

In [ ]:
probs = np.array([0.95, 0.80, 0.65, 0.55, 0.45, 0.30, 0.10])
labels = np.array([1,    1,    0,    1,    0,    1,    0])

for threshold in [0.3, 0.5, 0.7]:
    preds = (probs >= threshold).astype(int)

    tp, tn, fp, fn = confusion_counts(
        labels,
        preds
    )

    acc, prec, rec, f1 = classification_metrics(
        tp, tn, fp, fn
    )

    print(
        f"threshold={threshold:.1f} | "
        f"precision={prec:.3f} | "
        f"recall={rec:.3f} | "
        f"F1={f1:.3f}"
    )

### 직접 해보기 4 — Threshold와 trade-off

`threshold`를 낮추거나 높여 보세요.

### 질문
1. threshold를 낮추면 positive prediction은 많아집니까, 적어집니까?
2. recall은 어떤 방향으로 변하는 경향이 있습니까?
3. precision은 항상 recall과 같은 방향으로 변합니까?

## 16. Imbalanced Data

강의의 예제처럼 다음 데이터가 있다고 생각해 봅니다.

- Actual Positive = 5
- Actual Negative = 9,995
- 모델은 모든 sample을 negative로 예측

그러면

- TP = 0
- FP = 0
- FN = 5
- TN = 9,995

입니다.

In [ ]:
tp = 0
fp = 0
fn = 5
tn = 9995

accuracy, precision, recall, f1 = classification_metrics(
    tp, tn, fp, fn
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1}")

### 확인할 내용

이 모델은 거의 모든 데이터를 맞추므로 accuracy는 매우 높게 나옵니다.

하지만 positive sample을 하나도 찾지 못합니다.

따라서 **imbalanced data에서는 accuracy 하나만으로 모델을 평가하면 안 됩니다.**

특히 의료 진단처럼 positive를 놓치면 큰 문제가 되는 상황에서는
recall이 매우 중요할 수 있습니다.

# 17. 최종 실습

### 기본
1. sigmoid 함수에 `z = -5, 0, 5`를 넣고 출력값을 비교하세요.
2. `a`, `b`를 바꾸어 sigmoid curve의 위치와 기울기를 관찰하세요.
3. BCE에서 정답과 반대 방향으로 probability를 높였을 때 loss가 어떻게 변하는지 확인하세요.
4. 직접 구현한 logistic regression에서 learning rate를 바꾸어 convergence를 비교하세요.

### TensorFlow/Keras
5. Keras logistic regression의 학습된 weight와 bias를 출력하세요.
6. 새로운 study time을 입력하고 pass probability를 예측하세요.
7. softmax model의 두 output probability 합이 항상 1인지 확인하세요.

### Cross Entropy
8. one-hot target과 두 개의 prediction distribution을 직접 만들고 CCE를 비교하세요.
9. entropy가 높은 distribution과 낮은 distribution을 각각 하나씩 만들어 비교하세요.

### Performance Metrics
10. 임의의 `y_true`, `y_pred`를 만들고 TP/TN/FP/FN을 직접 계산하세요.
11. threshold를 바꾸면서 precision과 recall의 trade-off를 관찰하세요.
12. imbalanced data에서 accuracy가 높아도 좋은 classifier가 아닐 수 있는 예를 설명하세요.

### 도전
아래 함수들을 직접 구현해 보세요.

```python
def my_sigmoid(z):
    pass

def my_binary_cross_entropy(y_true, y_prob):
    pass

def my_softmax(z):
    pass

def my_accuracy(tp, tn, fp, fn):
    pass

def my_precision(tp, fp):
    pass

def my_recall(tp, fn):
    pass
```

# 18. 정리

이번 실습에서는 classification의 기본 흐름을 하나의 구조로 연결했습니다.

| 개념 | 역할 |
|---|---|
| Logit | 모델이 만든 raw score |
| Sigmoid | binary class probability |
| Softmax | multiclass probability distribution |
| BCE | binary classification loss |
| CCE | multiclass classification loss |
| Entropy | probability distribution의 불확실성 |
| Cross Entropy | target과 prediction의 mismatch |
| Confusion Matrix | TP, TN, FP, FN 정리 |
| Accuracy | 전체 중 맞춘 비율 |
| Precision | positive prediction 중 실제 positive 비율 |
| Recall | 실제 positive 중 찾아낸 비율 |
| F1 Score | precision과 recall의 조화평균 |

### 꼭 기억할 것

Binary logistic regression의 핵심 구조는

$$
z = ax+b
$$

$$
\hat{y}
=
\sigma(z)
$$

입니다.

즉,

**입력 → linear score(logit) → sigmoid → probability**

의 흐름입니다.

Multiclass classification에서는 이 구조가

**입력 → logits → softmax → probability distribution**

으로 확장됩니다.

그리고 training은 ground truth와 prediction 사이의
cross entropy loss를 줄이는 방향으로 model parameter를 업데이트하는 과정입니다.